<a href="https://colab.research.google.com/github/BilalAsifB/MissenseImpact-FYP/blob/main/notebooks/data-preprocessing/Bilal/process_data_indigen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install cyvcf2 --quiet
!apt-get update
!apt-get install -y bcftools tabix

Hit:1 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [ ]:
import pandas as pd
import numpy as np
import datetime
import os
import subprocess
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from cyvcf2 import VCF

In [ ]:
# ====================  FILE PATHS ====================
input_vcf = '/content/drive/MyDrive/FYP_DATA/raw_data/IndiGen/IndiGen.vcf'
reference_fasta = '/content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa'

# Output in processed_data/IndiGen
output_dir = '/content/drive/MyDrive/FYP_DATA/processed_data/IndiGen'
output_vcf = f'{output_dir}/indigen_normalized.vcf.gz'

# Temporary files
temp_chr_renamed = f'{output_dir}/temp_chr_renamed.vcf.gz'
temp_filtered = f'{output_dir}/temp_filtered.vcf.gz'
# =========================================================

In [ ]:
print("\n" + "=" * 70)
print("IndiGen VCF NORMALIZATION PIPELINE")
print("=" * 70)
print(f"\n📥 Input VCF:  {input_vcf}")
print(f"📚 Reference:  {reference_fasta}")
print(f"📂 Output dir: {output_dir}")

# ============================================================
# STEP 1A: Fix VCF header by adding contig definitions
# ============================================================
print("\n" + "=" * 70)
print("STEP 1A/6: Fixing VCF header (adding contig definitions)...")
print("=" * 70)

temp_fixed = f'{output_dir}/temp_header_fixed.vcf'

print("📝 Reading VCF and adding proper contig definitions...")

# Read the original VCF and fix the header
with open(input_vcf, 'r') as infile, open(temp_fixed, 'w') as outfile:
    header_written = False

    for line in infile:
        if line.startswith('##'):
            # Write header lines as-is
            outfile.write(line)
        elif line.startswith('#CHROM'):
            # Before writing the #CHROM line, add contig definitions for chr 1-22
            if not header_written:
                print("   Adding contig definitions for chromosomes 1-22...")
                for i in range(1, 23):
                    outfile.write(f'##contig=<ID={i},length=999999999>\n')
                # Also add other common contigs that might be in the file
                outfile.write(f'##contig=<ID=X,length=999999999>\n')
                outfile.write(f'##contig=<ID=Y,length=999999999>\n')
                outfile.write(f'##contig=<ID=MT,length=999999999>\n')
                header_written = True
            # Now write the #CHROM line
            outfile.write(line)
        else:
            # Write data lines as-is
            outfile.write(line)

print("✓ VCF header fixed")

# Compress the fixed VCF
print("\n📦 Compressing fixed VCF...")
temp_fixed_gz = f'{output_dir}/temp_header_fixed.vcf.gz'

compress_cmd = f"bgzip -c {temp_fixed} > {temp_fixed_gz}"
result = subprocess.run(compress_cmd, shell=True, capture_output=True, text=True)

if result.returncode != 0:
    print("❌ Compression failed!")
    print(result.stderr)
    raise Exception("Compression failed")

print("✓ VCF compressed")

# Index the compressed file
print("\n📑 Indexing fixed VCF...")
index_result = subprocess.run(
    f"tabix -f -p vcf {temp_fixed_gz}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"❌ Indexing failed!")
    print(f"Error: {index_result.stderr}")
    raise Exception("Failed to create index")

print("✓ Index created")

# Remove uncompressed temp file
if os.path.exists(temp_fixed):
    os.remove(temp_fixed)
    print("✓ Cleaned up uncompressed temp file")


IndiGen VCF NORMALIZATION PIPELINE

📥 Input VCF:  /content/drive/MyDrive/FYP_DATA/raw_data/IndiGen/IndiGen.vcf
📚 Reference:  /content/drive/MyDrive/FYP_DATA/raw_data/reference_fasta/hg38_dna.fa
📂 Output dir: /content/drive/MyDrive/FYP_DATA/processed_data/IndiGen

STEP 1A/6: Fixing VCF header (adding contig definitions)...
📝 Reading VCF and adding proper contig definitions...
   Adding contig definitions for chromosomes 1-22...
✓ VCF header fixed

📦 Compressing fixed VCF...
✓ VCF compressed

📑 Indexing fixed VCF...
✓ Index created
✓ Cleaned up uncompressed temp file


In [ ]:
# ============================================================
# STEP 1B: Add "chr" prefix to chromosome names (1-22 only)
# ============================================================
print("\n" + "=" * 70)
print("STEP 1B/6: Adding 'chr' prefix to chromosome names (1-22 only)...")
print("=" * 70)

# Create chromosome rename file - ONLY for chromosomes 1-22
chr_rename_file = '/tmp/chr_rename_indigen.txt'
with open(chr_rename_file, 'w') as f:
    for i in range(1, 23):
        f.write(f"{i} chr{i}\n")

print("📝 Renaming: 1→chr1, 2→chr2, ..., 22→chr22 (ONLY main chromosomes)")

rename_cmd = f"""
bcftools annotate \
    --rename-chrs {chr_rename_file} \
    -O z \
    -o {temp_chr_renamed} \
    {temp_fixed_gz}
"""

print("⏱️  Renaming chromosomes...")
result = subprocess.run(
    rename_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ CHROMOSOME RENAMING FAILED!")
    print(result.stderr)
    raise Exception("Chromosome renaming failed")

print("✓ Chromosomes renamed")

# ============================================================
# STEP 2: Index renamed file
# ============================================================
print("\n📑 Creating index for renamed file...")
index_result = subprocess.run(
    f"tabix -f -p vcf {temp_chr_renamed}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"❌ Indexing failed!")
    print(f"Error: {index_result.stderr}")
    raise Exception("Failed to create index")
else:
    print(f"✓ Index created")

# Verify index exists
if os.path.exists(f"{temp_chr_renamed}.tbi"):
    index_size = os.path.getsize(f"{temp_chr_renamed}.tbi") / 1024
    print(f"✓ Index file verified ({index_size:.1f} KB)")
else:
    print("❌ Index file not found!")
    raise Exception("Failed to create index")



STEP 1B/6: Adding 'chr' prefix to chromosome names (1-22 only)...
📝 Renaming: 1→chr1, 2→chr2, ..., 22→chr22 (ONLY main chromosomes)
⏱️  Renaming chromosomes...
✓ Chromosomes renamed

📑 Creating index for renamed file...
✓ Index created
✓ Index file verified (1654.2 KB)


In [ ]:
# ============================================================
# STEP 3: Filter to keep only standard chromosomes (chr1-chr22)
# ============================================================
print("\n" + "=" * 70)
print("STEP 2/5: Filtering for chromosomes chr1-chr22 only...")
print("=" * 70)

# Create list of chromosomes to keep - these should now have chr prefix
chromosomes_to_keep = ','.join([f'chr{i}' for i in range(1, 23)])

print(f"📝 Keeping only: {chromosomes_to_keep[:50]}... (chr1-chr22)")

filter_cmd = f"""
bcftools view \
    -t {chromosomes_to_keep} \
    -O z \
    -o {temp_filtered} \
    {temp_chr_renamed}
"""

print("⏱️  Filtering to chr1-chr22...")
result = subprocess.run(
    filter_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print("\n❌ FILTERING FAILED!")
    print(result.stderr)
    # Print some debug info
    print("\nTrying to check what chromosomes are in the file...")
    check_cmd = f"bcftools view -H {temp_chr_renamed} | cut -f1 | sort -u | head -20"
    check_result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
    print("First 20 chromosomes found:")
    print(check_result.stdout)
    raise Exception("Filtering failed")

print("✓ Filtered to chromosomes chr1-chr22")

# ============================================================
# STEP 4: Index filtered file
# ============================================================
print("\n📑 Creating index for filtered file...")
index_result = subprocess.run(
    f"tabix -f -p vcf {temp_filtered}",
    shell=True,
    capture_output=True,
    text=True
)

if index_result.returncode != 0:
    print(f"❌ Indexing failed!")
    print(f"Error: {index_result.stderr}")
else:
    print(f"✓ Index created")

# ============================================================
# STEP 5: Normalize VCF
# ============================================================
print("\n" + "=" * 70)
print("STEP 3/5: Normalizing VCF...")
print("=" * 70)

norm_cmd = f"""
bcftools norm \
    -m- \
    -f {reference_fasta} \
    --check-ref w \
    -O z \
    -o {output_vcf} \
    {temp_filtered}
"""

print("⏱️  Normalizing (splitting multi-allelic, left-aligning)...")
result = subprocess.run(
    norm_cmd,
    shell=True,
    capture_output=True,
    text=True
)

# Print normalization stats
if result.stderr:
    print("\n📊 Normalization Stats:")
    print(result.stderr)

if result.returncode != 0:
    print("\n❌ NORMALIZATION FAILED!")
    print(result.stdout)
    print(result.stderr)
    raise Exception("Normalization failed")

print(f"\n✓ Normalized VCF created successfully!")

# ============================================================
# STEP 6: Index normalized VCF
# ============================================================
print("\n" + "=" * 70)
print("STEP 4/5: Indexing normalized VCF...")
print("=" * 70)

index_cmd = f"tabix -f -p vcf {output_vcf}"

result = subprocess.run(
    index_cmd,
    shell=True,
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(f"✓ Index created: {output_vcf}.tbi")
else:
    print("❌ Indexing failed!")
    print(result.stderr)
    raise Exception("Indexing failed")

# ============================================================
# STEP 7: Verify and cleanup
# ============================================================
print("\n" + "=" * 70)
print("STEP 5/5: Verifying normalization...")
print("=" * 70)

vcf_norm = VCF(output_vcf)

# Check chromosomes present
chromosomes_found = set()
multi_count = 0
total = 0
vrt_types = {}

print("\n⏱️  Scanning variants...")

for variant in vcf_norm:
    chromosomes_found.add(variant.CHROM)
    if len(variant.ALT) > 1:
        multi_count += 1

    # Count VRT types
    vrt = variant.INFO.get('VRT', None)
    if vrt:
        vrt_types[vrt] = vrt_types.get(vrt, 0) + 1

    total += 1

print(f"\n✓ Verification Results:")
print(f"   Total variants checked: {total:,}")
print(f"   Chromosomes found: {sorted(chromosomes_found)}")
print(f"   Multi-allelic variants: {multi_count}")

if multi_count == 0:
    print("   ✓ All variants are properly split!")
else:
    print(f"   ⚠️  Found {multi_count} multi-allelic variants")

# Print VRT distribution
print(f"\n📊 Variant Types (VRT) Distribution:")
vrt_labels = {
    1: "SNV (Single Nucleotide Variation)",
    2: "DIV (Deletion/Insertion)",
    3: "HETEROZYGOUS",
    4: "STR (Short Tandem Repeat)",
    5: "NAMED (Named Repetitive Element)",
    6: "NO VARIATION",
    7: "MIXED",
    8: "MNV (Multiple Nucleotide Variation)",
    9: "Exception"
}

for vrt_code in sorted(vrt_types.keys()):
    count = vrt_types[vrt_code]
    label = vrt_labels.get(vrt_code, f"Unknown ({vrt_code})")
    percentage = (count / total) * 100
    print(f"   VRT={vrt_code} ({label}): {count:,} ({percentage:.2f}%)")

# ============================================================
# STEP 8: Cleanup temporary files
# ============================================================
print("\n🧹 Cleaning up temporary files...")
temp_files = [
    temp_fixed_gz,
    temp_fixed_gz + '.tbi',
    temp_chr_renamed,
    temp_chr_renamed + '.tbi',
    temp_filtered,
    temp_filtered + '.tbi',
    chr_rename_file
]

cleaned_count = 0
for temp_file in temp_files:
    if os.path.exists(temp_file):
        try:
            os.remove(temp_file)
            cleaned_count += 1
            print(f"   Removed: {os.path.basename(temp_file)}")
        except Exception as e:
            print(f"   Could not remove {os.path.basename(temp_file)}: {e}")

print(f"\n✓ Cleaned up {cleaned_count} temporary files")

# ============================================================
# FINAL SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("✅ IndiGen VCF NORMALIZATION COMPLETE!")
print("=" * 70)

print(f"\n📂 Output directory:")
print(f"   {output_dir}")
print(f"\n📂 Normalized VCF file:")
print(f"   {os.path.basename(output_vcf)}")
print(f"\n📂 Index file:")
print(f"   {os.path.basename(output_vcf)}.tbi")

# Get file size
if os.path.exists(output_vcf):
    file_size = os.path.getsize(output_vcf) / (1024 * 1024)
    print(f"\n📊 File size: {file_size:.1f} MB")

print(f"\n📊 Summary:")
print(f"   • Contains only chromosomes 1-22")
print(f"   • All variants normalized and left-aligned")
print(f"   • Multi-allelic variants split")
print(f"   • Total variants: {total:,}")
print(f"\n📁 All IndiGen processed files are in:")
print(f"   {output_dir}")
print("=" * 70)

# ============================================================
# EXTRACT VCF TO DATAFRAME
# ============================================================

print("\n\n" + "=" * 70)
print("EXTRACTING IndiGen VCF TO DATAFRAME")
print("=" * 70)

normalized_vcf = output_vcf
print(f"\n📥 Input: {normalized_vcf}")

def extract_indigen_vcf_to_dataframe(vcf_path):
    """
    Extract IndiGen VCF to pandas DataFrame

    Args:
        vcf_path: path to normalized IndiGen VCF file

    Returns:
        pandas DataFrame with variant information
    """

    variants = []
    count = 0
    print("\n⏱️  Extracting data from VCF...")

    vcf = VCF(vcf_path)

    for variant in vcf:
        # Basic variant info
        chrom = variant.CHROM
        pos = variant.POS
        ref = variant.REF
        alt = variant.ALT[0] if variant.ALT else None

        # Extract INFO fields - IndiGen specific
        vrt = variant.INFO.get('VRT', None)  # Variation type

        # Store in list
        variants.append({
            'CHROM': chrom,
            'POS': pos,
            'REF': ref,
            'ALT': alt,
            'VRT': vrt
        })

        count += 1
        if count % 50000 == 0:
            print(f"   {count:,} variants extracted...")

    # Convert to DataFrame
    df = pd.DataFrame(variants)
    print(f"\n✓ Extraction complete!")
    print(f"   Total variants: {len(df):,}")

    return df

# Extract to DataFrame
indigen_df = extract_indigen_vcf_to_dataframe(normalized_vcf)

# ============================================================
# SAVE TO CSV
# ============================================================
print("\n💾 Saving DataFrame to CSV...")
csv_output = f'{output_dir}/indigen_normalized_variants.csv'
indigen_df.to_csv(csv_output, index=False)

csv_size = os.path.getsize(csv_output) / (1024 * 1024)
print(f"✓ Saved to: {csv_output}")
print(f"   File size: {csv_size:.1f} MB")

# ============================================================
# DATAFRAME SUMMARY
# ============================================================
print("\n" + "=" * 70)
print("DATAFRAME SUMMARY")
print("=" * 70)

print(f"\n📊 Shape: {indigen_df.shape[0]:,} rows × {indigen_df.shape[1]} columns")
print(f"\n📋 Columns: {list(indigen_df.columns)}")

print(f"\n📈 Chromosome Distribution:")
chrom_counts = indigen_df['CHROM'].value_counts().sort_index()
for chrom in chrom_counts.index[:5]:  # Show first 5
    print(f"   {chrom}: {chrom_counts[chrom]:,} variants")
print(f"   ... (showing first 5 of {len(chrom_counts)} chromosomes)")

print(f"\n📈 VRT Distribution:")
vrt_counts = indigen_df['VRT'].value_counts().sort_index()
vrt_labels = {
    1: "SNV", 2: "DIV", 3: "HETEROZYGOUS", 4: "STR",
    5: "NAMED", 6: "NO VARIATION", 7: "MIXED", 8: "MNV", 9: "Exception"
}
for vrt_code in sorted(vrt_counts.index):
    label = vrt_labels.get(vrt_code, f"Unknown")
    count = vrt_counts[vrt_code]
    percentage = (count / len(indigen_df)) * 100
    print(f"   VRT={vrt_code} ({label}): {count:,} ({percentage:.2f}%)")

print(f"\n👀 First few rows:")
print(indigen_df.head(10))

print("\n" + "=" * 70)
print("✅ IndiGen PROCESSING COMPLETE!")
print("=" * 70)
print(f"\n📁 Files created:")
print(f"   1. Normalized VCF: {output_vcf}")
print(f"   2. VCF Index: {output_vcf}.tbi")
print(f"   3. CSV file: {csv_output}")
print("=" * 70)


STEP 2/5: Filtering for chromosomes chr1-chr22 only...
📝 Keeping only: chr1,chr2,chr3,chr4,chr5,chr6,chr7,chr8,chr9,chr10... (chr1-chr22)
⏱️  Filtering to chr1-chr22...
✓ Filtered to chromosomes chr1-chr22

📑 Creating index for filtered file...
✓ Index created

STEP 3/5: Normalizing VCF...
⏱️  Normalizing (splitting multi-allelic, left-aligning)...

📊 Normalization Stats:
Lines   total/split/realigned/skipped:	16266617/0/11/0


✓ Normalized VCF created successfully!

STEP 4/5: Indexing normalized VCF...
✓ Index created: /content/drive/MyDrive/FYP_DATA/processed_data/IndiGen/indigen_normalized.vcf.gz.tbi

STEP 5/5: Verifying normalization...

⏱️  Scanning variants...

✓ Verification Results:
   Total variants checked: 16,266,617
   Chromosomes found: ['chr1', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr2', 'chr20', 'chr21', 'chr22', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9']
   Multi-allelic variants: 0
   ✓ All variants ar